# TFT Local Training Pipeline – GC (Gold Futures), 1-Minute Candles

Trains a TemporalFusionTransformer on GC 1-minute candles (M1, OHLCV plus order-flow columns from the ATAS export), stores the best checkpoint **locally** (no GCS / BigQuery anymore)
and then tests the model on a separate test CSV. The forecasts (several horizons) are appended as extra columns to the test CSV.

## 0. Configuration

In [ ]:
import os
os.environ.pop('GOOGLE_APPLICATION_CREDENTIALS', None)

In [ ]:
import torch
print(torch.__version__)               # sollte auf +cu128 enden
print(torch.cuda.is_available())       # True
print(torch.cuda.get_device_name(0))   # NVIDIA GeForce RTX 5080

In [ ]:
# ===== CONFIGURE THESE =====
from pathlib import Path

SYMBOL = "GC"  # Instrument represented by the training data
DATA_CSV_RELATIVE_PATH = (
    Path("data")
    / "Candelexport_GC_M1_25_11_2018_20_10_2026"
    / "E:\Programierung\Daten\Candelexport_GC_M1_25_11_2018_20_10_2026.csv"
)
DATA_CSV_CANDIDATES = [
    root / DATA_CSV_RELATIVE_PATH
    for root in (Path.cwd(), *Path.cwd().parents)
]
DATA_CSV_PATH = next(
    (path.resolve() for path in DATA_CSV_CANDIDATES if path.is_file()),
    None,
)
if DATA_CSV_PATH is None:
    searched_paths = ", ".join(str(path) for path in DATA_CSV_CANDIDATES)
    raise FileNotFoundError(f"Training CSV not found. Searched: {searched_paths}")

# Training hyperparameters
MAX_EPOCHS = 20            # upper limit – early stopping ends earlier if val_loss does not improve for PATIENCE epochs
BATCH_SIZE = 256             # was 64; raise only if the speed check after the first steps shows enough free VRAM
LEARNING_RATE = 0.0015      # raised together with the batch size (was 0.001)
HIDDEN_SIZE = 64
ATTENTION_HEAD_SIZE = 4
DROPOUT = 0.1
HIDDEN_CONTINUOUS_SIZE = 32
PATIENCE = 5                # Early stopping: stop after this many epochs without val_loss improvement
TRACK_EPOCH_METRICS = True  # per-epoch quality: MAE vs. naive forecast + direction hit rate (no extra compute)

# Sliding-window stride: 1 = every 1-minute window (2.2M training samples), 5 = every 5th window (~5x faster epochs).
# Neighbouring windows share 59 of 60 candles. Keep VAL_WINDOW_STRIDE identical between runs you want to compare.
TRAIN_WINDOW_STRIDE = 5   # 5 = quick check; set to 1 for the full comparison run
VAL_WINDOW_STRIDE = 1

# Measures before training whether data loading or the GPU computation limits the speed (~1 minute)
RUN_BOTTLENECK_CHECK = True

# Level-independent training: the model must not learn absolute gold price levels (they differ between
# training and validation/test periods). Target and price-level inputs are normalised per 60-minute window.
EXCLUDED_FEATURES = [
    "vwap", "obv", "vpt",                              # cumulative since file start -> meaningless in another period
    # CSV columns that are NOT used as model inputs (they stay unchanged in the CSV files)
    "Delta/Volume", "Session Delta", "Session Delta %",
    "Volume/sec", "Session Volume",
    "Height", "Time local", "Duration",
]
PRICE_LEVEL_COLUMNS = ["open", "high", "low", "Candel MaxVolumePrice", "Candel VAH", "Candel VAL"]
PRICE_LEVEL_CORR_THRESHOLD = 0.98  # other inputs correlating this strongly with close are treated as price level too

# Model architecture
MAX_ENCODER_LENGTH = 60  # Input window: last 60 one-minute candles
# One separate model per forecast length (minutes). Each model is trained only on its own horizon, so the
# short-term model is not dominated by the much larger errors of long horizons.
MODEL_HORIZONS = [15, 60]
TRAIN_MODEL_HORIZONS = MODEL_HORIZONS      # e.g. [60] to (re)train only one of the models
TARGET_HORIZON_BARS = max(MODEL_HORIZONS)  # rows without a known close this far ahead are dropped

# Local model storage (replaces the former GCS upload)
MODEL_DIR = Path("models")


def model_latest_path(horizon: int) -> Path:
    """Latest saved model for one forecast length (minutes)."""
    return MODEL_DIR / f"tft_checkpoint_{SYMBOL}_h{horizon}_latest.ckpt"


TRAINING_LOG_PATH = MODEL_DIR / "training_log.csv"   # replaces the former BigQuery log

# Test / inference on a separate CSV (section 8)
TEST_CSV_PATH = Path(r"E:\Programierung\Daten\Test_GC_M1_20_10_2025_20_10_2026.csv")
# Output = copy of the test CSV with forecast columns appended.
# Set TEST_OUTPUT_CSV_PATH = TEST_CSV_PATH to write into the original file instead.
TEST_OUTPUT_CSV_PATH = TEST_CSV_PATH.with_name(TEST_CSV_PATH.stem + "_predictions.csv")
# Forecast columns (minutes ahead). Each column comes from the shortest model in MODEL_HORIZONS that covers it:
# +1..+15 from the 15-minute model, +30..+60 from the 60-minute model.
PREDICTION_HORIZONS = [1, 5, 10, 15, 30, 45, 60]
EXPORT_QUANTILES = [0.5]                  # 0.5 = median forecast; e.g. [0.1, 0.5, 0.9] adds a forecast band
PREDICT_BATCH_SIZE = 1024

# Signal evaluation in futures ticks (section 8.4)
TICK_SIZE = 0.10                # GC / MGC: minimum price step 0.10 USD
TICK_VALUE_USD = 10.0           # GC: 10 USD per tick and contract (MGC: 1 USD)
COST_PER_TRADE_TICKS = 2.0      # round trip incl. spread/slippage and commission, in ticks -> adjust to your account
SIGNAL_WINDOWS = [15, 60]       # minutes; a predicted target counts as reached as soon as it is touched within the window
SIGNAL_THRESHOLDS_TICKS = [10, 20, 30, 40, 50]   # trade only if the predicted move is at least this many ticks
STOP_LOSS_TICKS = 30            # stop loss against the entry; None = no stop (only target or time exit)
TARGET_FILL_EXTRA_TICKS = 1     # target counts as filled only if the price trades this many ticks beyond it
MAX_GAP_MINUTES = 5             # no trade if the window contains a pause longer than this (session break, weekend)
BASELINE_RANDOM_RUNS = 5        # comparison runs with random direction (same entries and targets as the model)

# Text columns with "%" that are converted to numeric features when loading the CSV
PERCENT_COLUMNS = ["Delta/Volume", "Session Delta %"]

# Use GPU if available
import torch
ACCELERATOR = "gpu" if torch.cuda.is_available() else "cpu"

# Performance settings – software only, no GPU overclocking
NUM_WORKERS = 4        # CPU processes that prepare training batches in parallel (automatic fallback to 0 if they fail)
VAL_NUM_WORKERS = 2    # same for validation / test data
# Full 32-bit precision. bf16 stores gold prices around 4000 USD only in steps of 16-32 USD, which made 1-minute
# forecasts useless (test on random-walk data: MAE/naive stuck at ~2.2 with bf16, ~1.0 with 32 bit).
# The TF32 tensor cores of the RTX 5080 are still used (torch.set_float32_matmul_precision).
PRECISION = "32-true"
print(f"Training on: {ACCELERATOR}")
print(f"Symbol: {SYMBOL}, Epochs: {MAX_EPOCHS}, models (forecast minutes): {MODEL_HORIZONS}")
print(f"Batch size: {BATCH_SIZE}, workers: {NUM_WORKERS}/{VAL_NUM_WORKERS}, precision: {PRECISION}")
print(f"DATA_CSV_PATH: {DATA_CSV_PATH}")
print(f"TEST_CSV_PATH: {TEST_CSV_PATH}")

## 1. Install / Import Dependencies

In [ ]:
%pip install --upgrade ipywidgets

In [ ]:
import gc
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor, ModelCheckpoint
from pytorch_forecasting import TimeSeriesDataSet, GroupNormalizer, TemporalFusionTransformer
from pytorch_forecasting.data import EncoderNormalizer
from pytorch_forecasting.metrics import QuantileLoss, MultiHorizonMetric
import sys
import subprocess
import contextlib
import json
import re
import shutil
import time

try:
    from IPython.display import display
except ImportError:  # outside Jupyter
    display = print

torch.set_float32_matmul_precision("medium")

print(f"PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}")
pd.set_option("display.max_columns", None)

In [ ]:
def make_dataloader(dataset, train: bool, batch_size: int, num_workers: int):
    """
    DataLoader with parallel CPU workers so the GPU does not wait for data.
    If worker processes fail (can happen with Windows + Jupyter), it falls back to num_workers=0.
    """
    pin = torch.cuda.is_available()
    if num_workers > 0:
        try:
            dl = dataset.to_dataloader(
                train=train, batch_size=batch_size, num_workers=num_workers,
                persistent_workers=True, pin_memory=pin,
            )
            next(iter(dl))  # start the workers once to check that multiprocessing works here
            print(f"DataLoader ({'train' if train else 'eval'}): {num_workers} workers, batch {batch_size}")
            return dl
        except Exception as exc:
            print(f"⚠ DataLoader with {num_workers} workers failed ({type(exc).__name__}: {exc}) -> using num_workers=0")
    print(f"DataLoader ({'train' if train else 'eval'}): 0 workers, batch {batch_size}")
    return dataset.to_dataloader(train=train, batch_size=batch_size, num_workers=0, pin_memory=pin)

## 2. Loss Function

Standard `QuantileLoss` from pytorch_forecasting with 7 quantiles. The 0.5 quantile (median) is the price forecast,
the outer quantiles form the forecast band. The former custom `DirectionAwareQuantileLoss` was built for direction
prediction and has been removed.

In [ ]:
LOSS_QUANTILES = [0.02, 0.1, 0.25, 0.5, 0.75, 0.9, 0.98]  # 0.5 = median = price forecast

### Data Load from the GC 1-Minute CSV

Each row is one GC 1-minute candle (M1) from the ATAS export. The CSV is loaded once and split into training and validation data.

In [ ]:
def load_gc_m1_csv(csv_path: str | Path = DATA_CSV_PATH, keep_bar: bool = False) -> pd.DataFrame:
    """Load all columns of a GC 1-minute candle CSV (ATAS export) in source Bar order."""
    raw = pd.read_csv(
        csv_path,
        sep=";",
        dtype={
            "Bar": "int64",
            "Open": "float32",
            "High": "float32",
            "Low": "float32",
            "Close": "float32",
            "Volume": "float32",
        },
        encoding="utf-8-sig",
    )
    if raw.empty:
        raise ValueError(f"No rows found in GC M1 CSV: {csv_path}")

    # Percent columns are exported as text ("59.15%") and would otherwise be silently ignored
    # as non-numeric. Convert them to fractions (59.15% -> 0.5915) so they become model features.
    for col in PERCENT_COLUMNS:
        if col not in raw.columns:
            continue
        text = (
            raw[col].astype(str).str.strip()
            .str.replace("%", "", regex=False)
            .str.replace("'", "", regex=False)
            .str.replace(" ", "", regex=False)
        )
        # Decimal comma only if the value has no decimal point (e.g. "59,15%")
        text = text.where(text.str.contains(".", regex=False), text.str.replace(",", ".", regex=False))
        values = pd.to_numeric(text, errors="coerce")
        n_bad = int((values.isna() & (text != "") & (text.str.lower() != "nan")).sum())
        if n_bad:
            print(f"⚠ {col}: {n_bad:,} values not parseable -> set to 0")
        raw[col] = (values / 100).fillna(0.0).astype("float32")

    required_values = ["Time UTC", "Open", "High", "Low", "Close", "Volume"]
    if raw[required_values].isna().any().any():
        raise ValueError(f"GC M1 CSV contains missing timestamps or OHLCV values: {csv_path}")

    # Preserve the source sequence: timestamps can occasionally reverse by a few seconds.
    if not raw["Bar"].is_monotonic_increasing:
        raw = raw.sort_values("Bar", kind="stable")

    df = raw.rename(columns={
        "Time UTC": "timestamp",
        "Open": "open",
        "High": "high",
        "Low": "low",
        "Close": "close",
        "Volume": "volume",
    })
    df["timestamp"] = pd.to_datetime(
        df["timestamp"], format="%d.%m.%Y %H:%M:%S", errors="raise", utc=True
    )
    # "Bar" is only the running counter of the export: used for ordering, never as model input.
    # keep_bar=True is only needed by the test section to map forecasts back to the CSV rows.
    if not keep_bar:
        df = df.drop(columns=["Bar"])
    df = df.reset_index(drop=True)
    print(f"Loaded {len(df):,} GC 1-minute candles from {df['timestamp'].min()} to {df['timestamp'].max()}")
    return df

In [ ]:
df_raw = load_gc_m1_csv()
LOOKBACK_DAYS = (df_raw["timestamp"].max() - df_raw["timestamp"].min()).days
df_raw = df_raw.drop(columns=["Time local"])
df_raw.head()

In [ ]:
df_raw.shape

In [ ]:
df_raw.columns

## 3. Loaded GC 1-Minute Candles

In [ ]:
print(f"CSV source: {DATA_CSV_PATH}")
print(f"Candles loaded: {len(df_raw):,}")
print(f"Date range (UTC): {df_raw['timestamp'].min()} to {df_raw['timestamp'].max()}")

In [ ]:
df_raw.tail()

## 4. Feature Engineering

In [ ]:
def calculate_all_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calculate all indicator features for the TFT model.
    Input df must have: timestamp, open, high, low, close, volume.
    All windows count GC 1-minute candles, so the minute suffixes (returns_5m, target_close_60m, ...) are real minutes
    of trading time (session breaks and weekends are skipped).
    """
    df = df.copy()

    # 1. RETURNS (5 features)
    df['returns_1m'] = df['close'].pct_change(1)
    df['returns_5m'] = df['close'].pct_change(5)
    df['returns_15m'] = df['close'].pct_change(15)
    df['returns_30m'] = df['close'].pct_change(30)
    df['returns_60m'] = df['close'].pct_change(60)

    # 2. PRICE RATIOS (6 features)
    df['high_low_ratio'] = df['high'] / df['low']
    df['close_open_ratio'] = df['close'] / df['open']
    df['high_close_ratio'] = df['high'] / df['close']
    df['low_close_ratio'] = df['low'] / df['close']
    df['upper_shadow'] = (df['high'] - np.maximum(df['open'], df['close'])) / (df['high'] - df['low'] + 1e-10)
    df['lower_shadow'] = (np.minimum(df['open'], df['close']) - df['low']) / (df['high'] - df['low'] + 1e-10)

    # 3. SMA FEATURES (11 features)
    for period in [5, 10, 20, 30]:
        sma = df['close'].rolling(window=period).mean()
        df[f'sma_{period}_slope'] = sma.pct_change()
        df[f'close_to_sma_{period}'] = (df['close'] - sma) / sma

    sma_60 = df['close'].rolling(window=60).mean()
    df['close_to_sma_60'] = (df['close'] - sma_60) / sma_60

    sma_120 = df['close'].rolling(window=120).mean()
    df['sma_120_slope'] = sma_120.pct_change()
    df['close_to_sma_120'] = (df['close'] - sma_120) / sma_120

    # 4. MACD (2 features)
    ema_12 = df['close'].ewm(span=12, adjust=False).mean()
    ema_26 = df['close'].ewm(span=26, adjust=False).mean()
    df['macd'] = ema_12 - ema_26
    signal_line = df['macd'].ewm(span=9, adjust=False).mean()
    df['macd_histogram'] = df['macd'] - signal_line

    # 5. VOLATILITY (4 features)
    df['volatility_5'] = df['returns_1m'].rolling(window=5).std()
    df['volatility_10'] = df['returns_1m'].rolling(window=10).std()
    df['volatility_20'] = df['returns_1m'].rolling(window=20).std()
    df['volatility_60'] = df['returns_1m'].rolling(window=60).std()

    # 6. ATR (2 features)
    high_low = df['high'] - df['low']
    high_close = np.abs(df['high'] - df['close'].shift())
    low_close = np.abs(df['low'] - df['close'].shift())
    tr = np.maximum(high_low, np.maximum(high_close, low_close))
    df['atr_14'] = tr.rolling(window=14).mean()
    df['atr_60'] = tr.rolling(window=60).mean()

    # 7. BOLLINGER BANDS (4 features)
    for period in [20, 60]:
        sma = df['close'].rolling(window=period).mean()
        std = df['close'].rolling(window=period).std()
        df[f'bb_std_{period}'] = std
        df[f'bb_position_{period}'] = (df['close'] - sma) / (2 * std + 1e-10)

    # 8. RSI (3 features)
    def calc_rsi(series, period):
        delta = series.diff()
        gain = delta.where(delta > 0, 0).rolling(window=period).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
        rs = gain / (loss + 1e-10)
        return 100 - (100 / (1 + rs))

    df['rsi_14'] = calc_rsi(df['close'], 14)
    df['rsi_20'] = calc_rsi(df['close'], 20)
    df['rsi_60'] = calc_rsi(df['close'], 60)

    # 9. STOCHASTIC (3 features)
    for period in [14, 60]:
        lowest_low = df['low'].rolling(window=period).min()
        highest_high = df['high'].rolling(window=period).max()
        df[f'stoch_k_{period}'] = 100 * (df['close'] - lowest_low) / (highest_high - lowest_low + 1e-10)
    df['stoch_d_14'] = df['stoch_k_14'].rolling(window=3).mean()

    # 10. RATE OF CHANGE (2 features)
    df['roc_10'] = df['close'].pct_change(10) * 100
    df['roc_20'] = df['close'].pct_change(20) * 100

    # 11. VOLUME INDICATORS (12 features)
    df['volume_change'] = df['volume'].pct_change(1)
    df['volume_change_5m'] = df['volume'].pct_change(5)

    for period in [5, 10, 20, 60]:
        df[f'volume_sma_{period}'] = df['volume'].rolling(window=period).mean()
        df[f'volume_ratio_{period}'] = df['volume'] / (df[f'volume_sma_{period}'] + 1e-10)

    # 12. OBV, VPT, MFI (5 features)
    obv = np.where(df['close'] > df['close'].shift(), df['volume'],
                   np.where(df['close'] < df['close'].shift(), -df['volume'], 0))
    df['obv'] = np.cumsum(obv)
    obv_sma = pd.Series(df['obv']).rolling(window=20).mean()
    df['obv_ratio'] = df['obv'] / (obv_sma + 1e-10)

    df['vpt'] = (df['volume'] * df['close'].pct_change()).cumsum()

    def calc_mfi(df, period):
        typical_price = (df['high'] + df['low'] + df['close']) / 3
        money_flow = typical_price * df['volume']
        positive_flow = money_flow.where(typical_price > typical_price.shift(), 0).rolling(window=period).sum()
        negative_flow = money_flow.where(typical_price < typical_price.shift(), 0).rolling(window=period).sum()
        mfi = 100 - (100 / (1 + positive_flow / (negative_flow + 1e-10)))
        return mfi

    df['mfi_14'] = calc_mfi(df, 14)
    df['mfi_60'] = calc_mfi(df, 60)

    # 13. VWAP RATIOS (2 features)
    vwap_20 = (df['volume'] * df['close']).rolling(window=20).sum() / (df['volume'].rolling(window=20).sum() + 1e-10)
    vwap_60 = (df['volume'] * df['close']).rolling(window=60).sum() / (df['volume'].rolling(window=60).sum() + 1e-10)
    df['close_to_vwap_20'] = (df['close'] - vwap_20) / vwap_20
    df['close_to_vwap_60'] = (df['close'] - vwap_60) / vwap_60

    # 14. TIME FEATURES (10 features)
    df['hour'] = df['timestamp'].dt.hour
    df['minute'] = df['timestamp'].dt.minute
    df['day_of_week'] = df['timestamp'].dt.dayofweek
    df['is_morning'] = ((df['hour'] >= 9) & (df['hour'] < 12)).astype(int)
    df['is_afternoon'] = ((df['hour'] >= 12) & (df['hour'] < 16)).astype(int)

    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    df['minute_sin'] = np.sin(2 * np.pi * df['minute'] / 60)
    df['minute_cos'] = np.cos(2 * np.pi * df['minute'] / 60)
    df['day_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['day_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)

    # 15. VOLUME LAGS (5 features)
    df['volume_lag_1'] = df['volume'].shift(1)
    df['volume_lag_5'] = df['volume'].shift(5)
    df['volume_lag_15'] = df['volume'].shift(15)
    df['volume_lag_30'] = df['volume'].shift(30)
    df['volume_lag_60'] = df['volume'].shift(60)

    # 15b. LEVEL-INDEPENDENT VERSIONS of indicators measured in USD (their size grows with the gold price)
    for col in ["macd", "macd_histogram", "atr_14", "atr_60", "bb_std_20", "bb_std_60"]:
        df[col] = df[col] / df["close"]
    if "Height" in df.columns:
        df["Height"] = df["Height"] / df["close"]

    # 16. TARGET VARIABLES (2 features)
    # Close 60 one-minute candles ahead (60 minutes of trading time).
    df['target_close_60m'] = df['close'].shift(-TARGET_HORIZON_BARS)
    df['target_return_60m'] = df['close'].pct_change(TARGET_HORIZON_BARS).shift(-TARGET_HORIZON_BARS)

    return df

In [ ]:
def remove_highly_correlated_features(df: pd.DataFrame, threshold: float = 0.95):
    """Remove features with pairwise correlation > threshold."""
    exclude_cols = [
        "timestamp", "time_idx", "group",
        "target_close_60m", "target_return_60m",
        "open", "high", "low", "close", "volume", "vwap",
    ] + EXCLUDED_FEATURES
    feature_cols = [
        c for c in df.columns
        if c not in exclude_cols and df[c].dtype in ["float64", "float32", "int64"]
    ]

    df_clean = df[feature_cols].dropna()
    corr_matrix = df_clean.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [col for col in upper.columns if any(upper[col] > threshold)]

    print(f"Correlation filtering (threshold={threshold}):")
    print(f"  Features before: {len(feature_cols)}")
    print(f"  Features dropped: {len(to_drop)} — {to_drop[:10]}{'...' if len(to_drop) > 10 else ''}")
    print(f"  Features remaining: {len(feature_cols) - len(to_drop)}")

    return df.drop(columns=to_drop), to_drop


def prepare_dataset(df: pd.DataFrame):
    """Run feature engineering and prepare the TFT-ready DataFrame."""
    # Calculate VWAP
    if df['volume'].sum() > 0:
        df["vwap"] = (df["volume"] * (df["high"] + df["low"] + df["close"]) / 3).cumsum() / df["volume"].cumsum()
    else:
        df["vwap"] = (df["high"] + df["low"] + df["close"]) / 3

    # Feature engineering
    df_feat = calculate_all_features(df)

    # Drop rows with NaN from indicators
    target_cols = ["target_close_60m", "target_return_60m"]
    non_target = [c for c in df_feat.columns if c not in target_cols]
    df_feat = df_feat.dropna(subset=non_target)
    df_feat = df_feat.dropna(subset=target_cols)
    df_feat = df_feat.reset_index(drop=True)

    # Correlation filtering
    df_feat, dropped_cols = remove_highly_correlated_features(df_feat, threshold=0.95)

    # Add required columns
    df_feat["time_idx"] = range(len(df_feat))
    df_feat["group"] = "default"

    # Clean infinities
    numeric_cols = df_feat.select_dtypes(include=[np.number]).columns.tolist()
    df_feat[numeric_cols] = df_feat[numeric_cols].replace([np.inf, -np.inf], np.nan)
    df_feat[numeric_cols] = df_feat[numeric_cols].ffill().bfill()

    print(f"\nPrepared dataset: {len(df_feat):,} rows, {len(df_feat.columns)} columns")
    return df_feat

In [ ]:
# Run feature engineering
df_tft = prepare_dataset(df_raw)

# Save metadata
data_start = str(df_raw['timestamp'].min())
data_end = str(df_raw['timestamp'].max())
print(f"Data range: {data_start} to {data_end}")

# Free raw data
del df_raw
gc.collect()

In [ ]:
df_tft.head()

## 5. Build TimeSeriesDataSets

In [ ]:
def build_feature_lists(df: pd.DataFrame):
    """Derive the time-varying known and unknown reals lists."""
    time_varying_known_reals = ["hour_sin", "hour_cos", "minute_sin", "minute_cos", "day_sin", "day_cos"]
    time_varying_known_reals = [c for c in time_varying_known_reals if c in df.columns]

    exclude = [
        "timestamp", "time_idx", "group",
        "target_close_60m", "target_return_60m",
        "hour", "minute", "day_of_week",
        "is_morning", "is_afternoon",
    ] + time_varying_known_reals + EXCLUDED_FEATURES

    time_varying_unknown_reals = [
        c for c in df.columns
        if c not in exclude and pd.api.types.is_numeric_dtype(df[c])
    ]

    if "close" in time_varying_unknown_reals:
        time_varying_unknown_reals.remove("close")
    time_varying_unknown_reals = ["close"] + time_varying_unknown_reals

    print(f"Time-varying known reals: {len(time_varying_known_reals)}")
    print(f"Time-varying unknown reals: {len(time_varying_unknown_reals)}")
    return time_varying_known_reals, time_varying_unknown_reals


time_varying_known_reals, time_varying_unknown_reals = build_feature_lists(df_tft)

# Train/validation split (80/20)
training_cutoff = int(len(df_tft) * 0.8)

print(f"\nDataset size: {len(df_tft):,} rows")
print(f"Training cutoff: time_idx={training_cutoff}")
print(f"Training rows: {len(df_tft[df_tft['time_idx'] <= training_cutoff]):,}")
print(f"Validation rows: {len(df_tft[df_tft['time_idx'] > training_cutoff]):,}")

In [ ]:
# Inputs that are absolute price levels -> normalised per window like the target
train_part = df_tft[df_tft["time_idx"] <= training_cutoff]
price_level_cols = [c for c in PRICE_LEVEL_COLUMNS if c in time_varying_unknown_reals]
for col in time_varying_unknown_reals:
    if col == "close" or col in price_level_cols:
        continue
    corr = train_part[col].corr(train_part["close"])
    if pd.notna(corr) and abs(corr) > PRICE_LEVEL_CORR_THRESHOLD:
        price_level_cols.append(col)
del train_part
print(f"Per-window normalised price-level inputs: {price_level_cols}")


def build_datasets(prediction_length: int):
    """Training and validation datasets for one forecast length (minutes)."""
    training = TimeSeriesDataSet(
        df_tft[lambda x: x.time_idx <= training_cutoff],
        time_idx="time_idx",
        target="close",
        group_ids=["group"],
        min_encoder_length=MAX_ENCODER_LENGTH // 2,
        max_encoder_length=MAX_ENCODER_LENGTH,
        max_prediction_length=prediction_length,
        static_categoricals=["group"],
        time_varying_known_categoricals=[],
        time_varying_known_reals=time_varying_known_reals,
        time_varying_unknown_categoricals=[],
        time_varying_unknown_reals=time_varying_unknown_reals,
        # Per-window normalisation: centre = LAST close of the encoder window, scale = std of the window
        # -> the model learns movements instead of absolute price levels; output 0 means "price stays".
        target_normalizer=EncoderNormalizer(max_length=[1, MAX_ENCODER_LENGTH]),
        scalers={col: EncoderNormalizer(max_length=[1, MAX_ENCODER_LENGTH]) for col in price_level_cols},
        add_relative_time_idx=True,
        add_target_scales=False,  # target centre/scale would reveal the absolute price level again
        add_encoder_length=True,
    )
    validation = TimeSeriesDataSet.from_dataset(
        training,
        df_tft[lambda x: x.time_idx > training_cutoff],
        predict=False,
    )
    # Sliding-window stride: keep only every n-th window (neighbouring 1-minute windows share 59 of 60 candles)
    if TRAIN_WINDOW_STRIDE > 1:
        training.filter(lambda idx: np.arange(len(idx)) % TRAIN_WINDOW_STRIDE == 0, copy=False)
    if VAL_WINDOW_STRIDE > 1:
        validation.filter(lambda idx: np.arange(len(idx)) % VAL_WINDOW_STRIDE == 0, copy=False)
    print(f"[{prediction_length} min] training samples: {len(training):,}, validation samples: {len(validation):,} "
          f"(window stride {TRAIN_WINDOW_STRIDE}/{VAL_WINDOW_STRIDE})")
    return training, validation

## 6. Train One Model per Forecast Length

For every entry in `TRAIN_MODEL_HORIZONS` (default: 15 and 60 minutes) a separate model is trained, evaluated, saved and logged.
The cells 6.1–6.3 only define helpers; cell 6.4 runs the training.

Per epoch a quality block is printed (prefixed with the model, e.g. `[15 min]`): MAE relative to the naive forecast
"price stays" (**< 1.00 = model is better**) and the direction hit rate (**> 50 % = better than a coin flip**).
The per-epoch CSV in `models/` is updated after every epoch.

### 6.1 Bottleneck Check (data loading vs. GPU compute)

In [ ]:
def run_bottleneck_check(model, dataloader, n_batches: int = 30, n_steps: int = 20) -> dict:
    """Samples/s of data loading vs. GPU compute (on a copy of the model)."""
    import copy

    dev = "cuda" if torch.cuda.is_available() else "cpu"

    # 1) Data loading only (DataLoader with its workers, no model)
    it = iter(dataloader)
    next(it)
    t0, n = time.time(), 0
    for i, (xb, yb) in enumerate(it):
        n += xb["encoder_cont"].shape[0]
        if i == n_batches - 1:
            break
    data_rate = n / (time.time() - t0)

    # 2) GPU compute only (one batch already on the GPU, forward + backward + optimizer step)
    def to_dev(v):
        if torch.is_tensor(v):
            return v.to(dev, non_blocking=True)
        if isinstance(v, (list, tuple)):
            return type(v)(to_dev(e) for e in v)
        return v

    x_gpu = {k: to_dev(v) for k, v in xb.items()}
    y_gpu = to_dev(yb)
    model_copy = copy.deepcopy(model).to(dev).train()
    opt = torch.optim.Adam(model_copy.parameters(), lr=1e-4)
    use_bf16 = PRECISION.startswith("bf16")

    def train_step():
        with torch.autocast(device_type=dev, dtype=torch.bfloat16, enabled=use_bf16):
            out = model_copy(x_gpu)
            loss = model_copy.loss(out["prediction"], y_gpu)
        loss.backward()
        opt.step()
        opt.zero_grad(set_to_none=True)

    for _ in range(3):
        train_step()
    if dev == "cuda":
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    for _ in range(n_steps):
        train_step()
    if dev == "cuda":
        torch.cuda.synchronize()
    gpu_rate = n_steps * xb["encoder_cont"].shape[0] / (time.time() - t0)

    print(f"Data loading ({NUM_WORKERS} workers, batch {xb['encoder_cont'].shape[0]}): {data_rate:8,.0f} samples/s")
    print(f"GPU compute  ({PRECISION}):                 {gpu_rate:8,.0f} samples/s")
    if dev == "cuda":
        free, total = torch.cuda.mem_get_info()
        print(f"VRAM: this test peak {torch.cuda.max_memory_reserved() / 1e9:.1f} GB, "
              f"used on card {(total - free) / 1e9:.1f} of {total / 1e9:.1f} GB")
    if data_rate < 0.8 * gpu_rate:
        print("→ Bottleneck = DATA LOADING: raise NUM_WORKERS (e.g. 12) – the GPU is waiting for batches.")
    else:
        print("→ Bottleneck = MODEL on the GPU: more workers will not help.")

    del model_copy, opt, x_gpu, y_gpu
    if dev == "cuda":
        torch.cuda.empty_cache()
    return {"data_samples_per_s": round(data_rate), "gpu_samples_per_s": round(gpu_rate)}

### 6.2 Callbacks: Per-Epoch Quality and Speed Check

In [ ]:
class EpochMetricsCallback(pl.Callback):
    """
    Collects per-epoch quality metrics on the validation data. The model output of the normal
    validation pass is captured with a forward hook, so there is no extra compute.
    """

    def __init__(self, horizons, max_length: int, csv_path=None, name: str = ""):
        super().__init__()
        self.csv_path = csv_path
        self.name = name
        self.horizons = [h for h in horizons if 1 <= h <= max_length]
        self.history = []
        self._acc = None
        self._val_done = False
        self._hook = None
        self._last_out = None

    def _capture_output(self, module, inputs, output):
        self._last_out = output

    def on_train_epoch_start(self, trainer, pl_module):
        self._t0 = time.time()
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()

    def on_validation_epoch_start(self, trainer, pl_module):
        self._acc = None
        self._val_done = False
        self._last_out = None
        if not trainer.sanity_checking and self._hook is None:
            self._hook = pl_module.register_forward_hook(self._capture_output)

    def on_validation_epoch_end(self, trainer, pl_module):
        if self._hook is not None:
            self._hook.remove()
            self._hook = None

    @torch.no_grad()
    def on_validation_batch_end(self, trainer, pl_module, outputs, batch, batch_idx, dataloader_idx=0):
        if trainer.sanity_checking or self._last_out is None:
            return
        x, y = batch
        out, self._last_out = self._last_out, None                         # output of the normal validation step
        pred = pl_module.to_prediction(out).float()                       # (batch, horizon) median, price scale
        actual = y[0].float()                                             # (batch, horizon)
        last_idx = (x["encoder_lengths"] - 1).long().unsqueeze(1)
        last_close = x["encoder_target"].float().gather(1, last_idx)[:, 0]

        h_idx = torch.tensor([h - 1 for h in self.horizons], device=pred.device)
        p, a, c = pred[:, h_idx], actual[:, h_idx], last_close.unsqueeze(1)
        moved = a != c
        stats = torch.stack([
            (p - a).abs().sum(0),                                              # model abs error
            (c - a).abs().sum(0),                                              # naive abs error
            ((torch.sign(p - c) == torch.sign(a - c)) & moved).sum(0).float(), # direction hits
            moved.sum(0).float(),                                              # bars with price change
            torch.full_like(p[0], p.shape[0]),                                 # samples
        ])
        self._acc = stats if self._acc is None else self._acc + stats
        self._val_done = True

    def on_train_epoch_end(self, trainer, pl_module):
        if not self._val_done or self._acc is None:
            return
        acc = self._acc.double().cpu().numpy()
        metrics = trainer.callback_metrics
        val = lambda k: float(metrics[k]) if k in metrics else float("nan")
        row = {
            "epoch": trainer.current_epoch + 1,
            "train_loss": val("train_loss_epoch"),
            "val_loss": val("val_loss"),
            "lr": trainer.optimizers[0].param_groups[0]["lr"],
            "minutes": (time.time() - getattr(self, "_t0", time.time())) / 60,
            "VRAM peak GB": torch.cuda.max_memory_reserved() / 1e9 if torch.cuda.is_available() else float("nan"),
        }
        for i, h in enumerate(self.horizons):
            row[f"MAE +{h}"] = acc[0, i] / acc[4, i]
            row[f"MAE naive +{h}"] = acc[1, i] / acc[4, i]
            row[f"MAE/naive +{h}"] = acc[0, i] / acc[1, i]
            row[f"Direction % +{h}"] = 100 * acc[2, i] / max(acc[3, i], 1)

        row["avg MAE/naive"] = float(np.mean([row[f"MAE/naive +{h}"] for h in self.horizons]))
        row["avg Direction %"] = float(np.mean([row[f"Direction % +{h}"] for h in self.horizons]))

        best_before = min((r["val_loss"] for r in self.history), default=float("inf"))
        row["new_best"] = row["val_loss"] < best_before
        self.history.append(row)
        if self.csv_path is not None:  # updated after every epoch -> can be opened while training runs
            pd.DataFrame(self.history).to_csv(self.csv_path, sep=";", index=False)

        best_epoch = min(self.history, key=lambda r: r["val_loss"])["epoch"]
        flag = "★ new best" if row["new_best"] else f"  (best: epoch {best_epoch})"
        ratio = " ".join(f"+{h} {row[f'MAE/naive +{h}']:.3f}" for h in self.horizons)
        direction = " ".join(f"+{h} {row[f'Direction % +{h}']:.1f}" for h in self.horizons)
        prev = self.history[-2] if len(self.history) > 1 else None
        trend = ""
        if prev is not None:
            d_ratio = row["avg MAE/naive"] - prev["avg MAE/naive"]
            d_dir = row["avg Direction %"] - prev["avg Direction %"]
            trend = (f"  vs. previous epoch: MAE/naive {d_ratio:+.3f} ({'better' if d_ratio < 0 else 'worse'}), "
                     f"direction {d_dir:+.1f} pp ({'better' if d_dir > 0 else 'worse'})")
        print(
            f"\n{self.name}Epoch {row['epoch']:>3}/{trainer.max_epochs} | train {row['train_loss']:.4f} | val {row['val_loss']:.4f} {flag} | "
            f"lr {row['lr']:.2e} | {row['minutes']:.1f} min\n"
            f"          MAE/naive  (<1 = better than naive): {ratio}   | avg {row['avg MAE/naive']:.3f}\n"
            f"          Direction % (>50 = better than coin): {direction}   | avg {row['avg Direction %']:.1f}\n"
            f"         {trend}"
        )


class GpuMemoryCheck(pl.Callback):
    """
    Early check after a few training steps: speed, estimated epoch duration and VRAM use.
    On Windows a full VRAM does not crash – the driver silently moves GPU memory to system RAM,
    which makes training many times slower (GPU at 99 % but low wattage).
    """

    def __init__(self, start_step: int = 5, check_step: int = 30):
        super().__init__()
        self.start_step, self.check_step = start_step, check_step
        self._t = None

    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx):
        if trainer.current_epoch != 0:
            return
        if batch_idx == self.start_step:
            self._t = time.time()
        elif batch_idx == self.check_step and self._t is not None:
            sec = (time.time() - self._t) / (self.check_step - self.start_step)
            batch_size = batch[0]["encoder_cont"].shape[0]
            msg = (f"\nSpeed check: {1 / sec:.2f} it/s = {batch_size / sec:,.0f} samples/s, "
                   f"~{sec * trainer.num_training_batches / 60:.0f} min per epoch (training part)")
            if torch.cuda.is_available():
                used = torch.cuda.max_memory_reserved() / 1e9
                total = torch.cuda.get_device_properties(0).total_memory / 1e9
                msg += f"\nVRAM: {used:.1f} of {total:.1f} GB"
                if used > 0.9 * total:
                    msg += ("\n⚠ VRAM (almost) full -> Windows swaps GPU memory to system RAM and training gets very slow. "
                            "Stop the run and lower BATCH_SIZE.")
                else:
                    msg += "\n→ VRAM ok (no swapping to system RAM)."
            print(msg)

### 6.3 Training, Evaluation, Saving and Logging of One Model

In [ ]:
def evaluate_model(model, val_dataloader) -> dict:
    """MAE / RMSE / MAPE / R² of the median forecast over all forecast steps of the validation set (USD)."""
    predictions = model.predict(
        val_dataloader, return_y=True, trainer_kwargs=dict(accelerator=ACCELERATOR, logger=False)
    )
    pred = predictions.output
    if pred.dim() == 3:
        pred = pred[..., LOSS_QUANTILES.index(0.5)]
    actual = predictions.y[0] if isinstance(predictions.y, (list, tuple)) else predictions.y
    pred_np = pred.detach().cpu().numpy().ravel()
    actual_np = actual.detach().cpu().numpy().ravel()
    errors = actual_np - pred_np
    ss_tot = np.sum((actual_np - actual_np.mean()) ** 2)
    return {
        "mae": float(np.mean(np.abs(errors))),
        "rmse": float(np.sqrt(np.mean(errors ** 2))),
        "mape": float(np.mean(np.abs(errors[actual_np != 0] / actual_np[actual_np != 0])) * 100),
        "r_squared": float(1 - np.sum(errors ** 2) / ss_tot) if ss_tot else None,
    }


def show_epoch_overview(horizon, epoch_cb, early_stop, trainer, best_model_path, csv_path):
    """Per-epoch table + chart + verdict whether more epochs would help."""
    epochs_trained = trainer.current_epoch  # Lightning 2: number of completed epochs
    match = re.search(r"epoch=(\d+)", Path(best_model_path).name)
    best_epoch = int(match.group(1)) + 1 if match else None

    epoch_df = pd.DataFrame(epoch_cb.history) if epoch_cb is not None else pd.DataFrame()
    if not epoch_df.empty:
        best_epoch = int(epoch_df.loc[epoch_df["val_loss"].idxmin(), "epoch"])
        epoch_df["val_loss change %"] = 100 * epoch_df["val_loss"].pct_change()
        epoch_df["best"] = np.where(epoch_df["epoch"] == best_epoch, "◀ best", "")
        epoch_df.to_csv(csv_path, sep=";", index=False)
        print(f"✓ Per-epoch metrics saved: {Path(csv_path).resolve()}")

    print(f"[{horizon} min] epochs trained: {epochs_trained} / {MAX_EPOCHS}, best epoch: {best_epoch}")
    if early_stop.stopped_epoch > 0:
        print(f"→ Early stopping: no val_loss improvement for {PATIENCE} epochs. More epochs would NOT help.")
    elif best_epoch is not None and best_epoch >= epochs_trained - 2:
        print("→ The best epoch is at the end of the run: the model was still improving. Raise MAX_EPOCHS.")
    else:
        print("→ MAX_EPOCHS reached, but the best epoch was earlier: more epochs would probably not help much.")

    if not epoch_df.empty:
        try:
            import matplotlib.pyplot as plt

            fig, axes = plt.subplots(1, 3, figsize=(16, 4))
            fig.suptitle(f"Model for {horizon}-minute forecasts")
            axes[0].plot(epoch_df["epoch"], epoch_df["train_loss"], marker="o", label="train")
            axes[0].plot(epoch_df["epoch"], epoch_df["val_loss"], marker="o", label="validation")
            axes[0].axvline(best_epoch, color="grey", linestyle="--", label="best epoch")
            axes[0].set_title("Loss")
            axes[0].legend()
            for h in epoch_cb.horizons:
                axes[1].plot(epoch_df["epoch"], epoch_df[f"MAE/naive +{h}"], marker=".", label=f"+{h} min")
                axes[2].plot(epoch_df["epoch"], epoch_df[f"Direction % +{h}"], marker=".", label=f"+{h} min")
            axes[1].axhline(1.0, color="red", linestyle="--")
            axes[1].set_title("MAE model / naive (< 1 = better than naive)")
            axes[1].legend(fontsize=8)
            axes[2].axhline(50.0, color="red", linestyle="--")
            axes[2].set_title("Direction correct % (> 50 = better than coin flip)")
            axes[2].legend(fontsize=8)
            for ax in axes:
                ax.set_xlabel("epoch")
            plt.tight_layout()
            plt.show()
        except ImportError:
            print("(matplotlib not installed – chart skipped)")
        display(epoch_df.round(4))
    return epochs_trained, best_epoch, epoch_df


def append_training_log(row: dict):
    """One row per trained model; keeps older rows even if the columns changed."""
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    log_df = pd.DataFrame([row])
    if TRAINING_LOG_PATH.exists():
        log_df = pd.concat([pd.read_csv(TRAINING_LOG_PATH, sep=";"), log_df], ignore_index=True)
    log_df.to_csv(TRAINING_LOG_PATH, sep=";", index=False)


def train_one_model(horizon: int, run_check: bool = False) -> dict:
    """Build datasets, train, evaluate, save and log the model for one forecast length (minutes)."""
    name = f"[{horizon} min] "
    print("=" * 80)
    print(f"MODEL FOR {horizon}-MINUTE FORECASTS")
    print("=" * 80)

    training, validation = build_datasets(horizon)
    train_dataloader = make_dataloader(training, train=True, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)
    val_dataloader = make_dataloader(validation, train=False, batch_size=BATCH_SIZE * 2, num_workers=VAL_NUM_WORKERS)

    model = TemporalFusionTransformer.from_dataset(
        training,
        learning_rate=LEARNING_RATE,
        hidden_size=HIDDEN_SIZE,
        attention_head_size=ATTENTION_HEAD_SIZE,
        dropout=DROPOUT,
        hidden_continuous_size=HIDDEN_CONTINUOUS_SIZE,
        output_size=len(LOSS_QUANTILES),
        loss=QuantileLoss(quantiles=LOSS_QUANTILES),
        reduce_on_plateau_patience=4,
    )
    print(f"{name}model parameters: {model.size() / 1e3:.1f}k")

    bottleneck = run_bottleneck_check(model, train_dataloader) if run_check else {}

    run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    checkpoint_dir = f"checkpoints/training/{SYMBOL}_h{horizon}_{run_timestamp}"
    os.makedirs(checkpoint_dir, exist_ok=True)
    early_stop = EarlyStopping(monitor="val_loss", min_delta=1e-4, patience=PATIENCE, verbose=True, mode="min")
    checkpoint_cb = ModelCheckpoint(
        dirpath=checkpoint_dir,
        monitor="val_loss",
        filename=f"tft-{SYMBOL}-h{horizon}" + "-{epoch:02d}-{val_loss:.4f}",
        save_top_k=1,
        mode="min",
    )
    callbacks = [early_stop, LearningRateMonitor(logging_interval="step"), checkpoint_cb, GpuMemoryCheck()]
    epoch_cb, epoch_csv = None, None
    if TRACK_EPOCH_METRICS:
        MODEL_DIR.mkdir(parents=True, exist_ok=True)
        epoch_csv = MODEL_DIR / f"epoch_metrics_{SYMBOL}_h{horizon}_stride{TRAIN_WINDOW_STRIDE}_{run_timestamp}.csv"
        print(f"{name}per-epoch metrics are written after every epoch to: {epoch_csv.resolve()}")
        epoch_cb = EpochMetricsCallback(PREDICTION_HORIZONS, max_length=horizon, csv_path=epoch_csv, name=name)
        callbacks.append(epoch_cb)

    trainer = pl.Trainer(
        max_epochs=MAX_EPOCHS,
        callbacks=callbacks,
        enable_model_summary=True,
        accelerator=ACCELERATOR,
        gradient_clip_val=0.1,
        precision=PRECISION,
    )
    print(f"{name}starting training on {ACCELERATOR} ({PRECISION}) ...")
    trainer.fit(model, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)

    best_model_path = checkpoint_cb.best_model_path
    best_val_loss = float(checkpoint_cb.best_model_score)
    print(f"{name}best model: {best_model_path} (val_loss {best_val_loss:.4f})")
    epochs_trained, best_epoch, epoch_df = show_epoch_overview(
        horizon, epoch_cb, early_stop, trainer, best_model_path, epoch_csv
    )

    best_model = TemporalFusionTransformer.load_from_checkpoint(best_model_path, strict=False)
    metrics = evaluate_model(best_model, val_dataloader)
    print(f"{name}validation (all forecast steps, USD): MAE {metrics['mae']:.4f}, RMSE {metrics['rmse']:.4f}")

    # Save: latest file per forecast length + timestamped copy that is never overwritten
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    timestamped_path = MODEL_DIR / f"tft_checkpoint_{SYMBOL}_h{horizon}_{run_timestamp}.ckpt"
    shutil.copy2(best_model_path, model_latest_path(horizon))
    shutil.copy2(best_model_path, timestamped_path)
    print(f"✓ Saved: {model_latest_path(horizon).resolve()}")
    print(f"✓ Saved: {timestamped_path.resolve()}")

    last = epoch_df.loc[epoch_df["epoch"] == best_epoch].iloc[0] if not epoch_df.empty else {}
    row = {
        "run_timestamp": datetime.now().isoformat(timespec="seconds"),
        "symbol": SYMBOL,
        "model_horizon_min": horizon,
        "best_val_loss": round(best_val_loss, 6),
        "mae": round(metrics["mae"], 6),
        "rmse": round(metrics["rmse"], 6),
        "mape": round(metrics["mape"], 6),
        "r_squared": round(metrics["r_squared"], 6) if metrics["r_squared"] is not None else None,
        "best_avg_mae_naive": round(float(last["avg MAE/naive"]), 4) if len(last) else None,
        "best_avg_direction_pct": round(float(last["avg Direction %"]), 2) if len(last) else None,
        "epochs_trained": epochs_trained,
        "best_epoch": best_epoch,
        "max_epochs": MAX_EPOCHS,
        "batch_size": BATCH_SIZE,
        "train_window_stride": TRAIN_WINDOW_STRIDE,
        "val_window_stride": VAL_WINDOW_STRIDE,
        "precision": PRECISION,
        "num_workers": NUM_WORKERS,
        "data_samples_per_s": bottleneck.get("data_samples_per_s"),
        "gpu_samples_per_s": bottleneck.get("gpu_samples_per_s"),
        "learning_rate": LEARNING_RATE,
        "hidden_size": HIDDEN_SIZE,
        "attention_head_size": ATTENTION_HEAD_SIZE,
        "dropout": DROPOUT,
        "patience": PATIENCE,
        "lookback_days": LOOKBACK_DAYS,
        "data_start_date": data_start,
        "data_end_date": data_end,
        "training_rows": len(training),
        "validation_rows": len(validation),
        "model_path": str(timestamped_path.resolve()),
    }
    append_training_log(row)
    print(f"✓ Logged to {TRAINING_LOG_PATH.resolve()}")

    del model, best_model, trainer, train_dataloader, val_dataloader, training, validation
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return row

### 6.4 Run Training for All Models

Trains the models in `TRAIN_MODEL_HORIZONS` one after the other. If you interrupt the run, the best checkpoint of the
interrupted model stays in `checkpoints/training/GC_h<minutes>_<time>/` and its per-epoch CSV in `models/`;
models that finished before are already saved and logged.

In [ ]:
training_results = {}
for i, horizon in enumerate(TRAIN_MODEL_HORIZONS):
    training_results[horizon] = train_one_model(horizon, run_check=(RUN_BOTTLENECK_CHECK and i == 0))

## 7. Training Summary

In [ ]:
summary_cols = ["model_horizon_min", "epochs_trained", "best_epoch", "best_val_loss",
                "best_avg_mae_naive", "best_avg_direction_pct", "mae", "model_path"]
summary_df = pd.DataFrame(training_results.values())[summary_cols]
print(f"Data range: {data_start} → {data_end}, window stride {TRAIN_WINDOW_STRIDE}, precision {PRECISION}")
display(summary_df)

## 8. Test on Separate CSV

Applies the saved models to the GC 1-minute test CSV `TEST_CSV_PATH`. For **every candle** each model gets the previous
`MAX_ENCODER_LENGTH` candles and forecasts the following closes. The columns in `PREDICTION_HORIZONS` are written behind the
original columns (`Pred_Close_+5` = forecast made at this candle for the close 5 one-minute candles later; session breaks and
weekends are skipped). Each column comes from the **shortest model that covers it**: +1…+15 from the 15-minute model,
+30…+60 from the 60-minute model.

* The first ~180 rows (≈ 3 hours) stay empty: indicator warm-up incl. SMA-120 plus the 60-candle input window.
* The feature columns are taken from each checkpoint, so the correlation filter is **not** re-run on the test data.
* **Testing without retraining** (e.g. after a kernel restart): run sections 0, 1 (imports + DataLoader helper), 2, the
  loader definition in "Data Load", the two definition cells in section 4 – then jump straight here.
  The models are loaded from `models/tft_checkpoint_GC_h<minutes>_latest.ckpt`.

In [ ]:
# ---- 8.1 Helpers ---------------------------------------------------------------------------
def build_inference_frame(df: pd.DataFrame, feature_cols: list[str], pad_bars: int) -> pd.DataFrame:
    """
    Same feature pipeline as prepare_dataset(), but
      * no correlation filter (feature list comes from the checkpoint),
      * rows without a future target are kept (we want forecasts up to the last candle),
      * `pad_bars` flat placeholder candles (+1 min each) are appended so the decoder of the last real candles has
        timestamps for the known time features. They never receive a forecast themselves.
    """
    df = df.copy().reset_index(drop=True)
    df["is_padding"] = False

    last = df.iloc[[-1]]
    step = df["timestamp"].diff().median()
    pad = pd.concat([last] * pad_bars, ignore_index=True)
    offsets = np.arange(1, pad_bars + 1)
    pad["timestamp"] = df["timestamp"].iloc[-1] + step * offsets
    pad["Bar"] = df["Bar"].iloc[-1] + offsets
    for col in ["open", "high", "low"]:
        pad[col] = last["close"].iloc[0]
    pad["is_padding"] = True
    df = pd.concat([df, pad], ignore_index=True)

    if df["volume"].sum() > 0:
        df["vwap"] = (df["volume"] * (df["high"] + df["low"] + df["close"]) / 3).cumsum() / df["volume"].cumsum()
    else:
        df["vwap"] = (df["high"] + df["low"] + df["close"]) / 3

    df_feat = calculate_all_features(df)

    missing = [c for c in feature_cols if c not in df_feat.columns]
    if missing:
        raise ValueError(f"Test CSV is missing feature columns the model was trained on: {missing}")

    df_feat = df_feat.dropna(subset=feature_cols).reset_index(drop=True)

    numeric_cols = [c for c in df_feat.select_dtypes(include=[np.number]).columns if c != "Bar"]
    df_feat[numeric_cols] = df_feat[numeric_cols].replace([np.inf, -np.inf], np.nan)
    df_feat[numeric_cols] = df_feat[numeric_cols].ffill().bfill()

    return df_feat.assign(time_idx=np.arange(len(df_feat), dtype="int64"), group="default")


def forecast_with_model(model_path, df_test_raw: pd.DataFrame, horizons: list[int]):
    """Rolling forecast of one saved model for every candle of the test data."""
    model = TemporalFusionTransformer.load_from_checkpoint(
        str(model_path), strict=False, map_location="cuda" if torch.cuda.is_available() else "cpu"
    )
    model.eval()
    params = model.dataset_parameters
    quantiles = [float(q) for q in model.loss.quantiles]
    enc_len, pred_len = params["max_encoder_length"], params["max_prediction_length"]
    feature_cols = [
        c for c in dict.fromkeys(params["time_varying_known_reals"] + params["time_varying_unknown_reals"])
        if c not in {"relative_time_idx", "encoder_length"}
    ]
    bad = [h for h in horizons if not 1 <= h <= pred_len]
    if bad:
        raise ValueError(f"{model_path}: horizons {bad} outside 1..{pred_len}")
    q_idx = {}
    for q in EXPORT_QUANTILES:
        hits = [i for i, mq in enumerate(quantiles) if abs(mq - q) < 1e-6]
        if not hits:
            raise ValueError(f"Quantile {q} not in model quantiles {quantiles}")
        q_idx[q] = hits[0]
    median_idx = quantiles.index(0.5)
    start_dependent = [c for c in ["Bar", "vwap", "obv", "vpt"] if c in feature_cols]
    if start_dependent:
        print(f"⚠ {model_path.name} uses start-dependent features {start_dependent} (old model?)")

    df_inf = build_inference_frame(df_test_raw, feature_cols, pad_bars=pred_len)
    n_real_rows = int((~df_inf["is_padding"]).sum())
    dataset = TimeSeriesDataSet.from_parameters(
        params, df_inf, predict=False, stop_randomization=True, min_encoder_length=enc_len
    )
    dataloader = make_dataloader(dataset, train=False, batch_size=PREDICT_BATCH_SIZE, num_workers=VAL_NUM_WORKERS)
    print(f"[{pred_len} min model] {model_path.name}: {len(dataset):,} forecasts for horizons {horizons}")
    predictions = model.predict(
        dataloader, mode="quantiles", return_index=True,
        trainer_kwargs=dict(accelerator=ACCELERATOR, logger=False),
    )
    q_out = predictions.output
    if q_out.dim() == 2:
        q_out = q_out.unsqueeze(-1)
    q_out = q_out.detach().cpu().numpy()                                 # (samples, horizon, quantiles)
    origin_idx = predictions.index["time_idx"].to_numpy() - 1            # last candle the model has seen
    assert not df_inf.loc[origin_idx, "is_padding"].any()

    pred_df = pd.DataFrame({"_bar": df_inf.loc[origin_idx, "Bar"].to_numpy().astype("int64")})
    for h in horizons:
        for q in EXPORT_QUANTILES:
            col = f"Pred_Close_+{h}" if q == 0.5 else f"Pred_Close_+{h}_q{round(q * 100):02d}"
            pred_df[col] = q_out[:, h - 1, q_idx[q]].astype("float64").round(2)

    close_arr = df_inf["close"].to_numpy(dtype="float64")
    eval_rows = []
    for h in horizons:
        target_idx = origin_idx + h
        ok = target_idx < n_real_rows                    # actual price must exist in the test file
        pred = q_out[ok, h - 1, median_idx].astype("float64")
        actual = close_arr[target_idx[ok]]
        now = close_arr[origin_idx[ok]]
        moved = actual != now
        eval_rows.append({
            "Horizon (min)": h,
            "Model": f"{pred_len} min",
            "Samples": int(ok.sum()),
            "MAE model": np.mean(np.abs(pred - actual)),
            "MAE naive (no change)": np.mean(np.abs(now - actual)),
            "MAE/naive": np.mean(np.abs(pred - actual)) / np.mean(np.abs(now - actual)),
            "Direction correct %": 100 * np.mean(np.sign(pred - now)[moved] == np.sign(actual - now)[moved]),
        })
    # Full median forecast path + actual prices for the signal evaluation (section 8.4)
    path = {
        "pred_len": pred_len,
        "median_path": q_out[:, :, median_idx].astype("float64"),
        "origin_idx": origin_idx,
        "close": close_arr,
        "high": df_inf["high"].to_numpy(dtype="float64"),
        "low": df_inf["low"].to_numpy(dtype="float64"),
        "timestamp": df_inf["timestamp"].to_numpy(),
        "n_real_rows": n_real_rows,
    }
    del model, dataset, dataloader
    return pred_df, eval_rows, path

In [ ]:
# ---- 8.2 Forecast with all models and write the CSV ----------------------------------------------
# Original file as text -> written back unchanged, only forecast columns are added
test_csv_text = pd.read_csv(TEST_CSV_PATH, sep=";", dtype=str, encoding="utf-8-sig", keep_default_na=False)
df_test_raw = load_gc_m1_csv(TEST_CSV_PATH, keep_bar=True).drop(columns=["Time local"], errors="ignore")
print(f"Test candles: {len(df_test_raw):,}  ({df_test_raw['timestamp'].min()} → {df_test_raw['timestamp'].max()})")

# Each forecast column comes from the shortest model whose forecast length covers it
horizon_to_model = {}
for h in PREDICTION_HORIZONS:
    covering = [m for m in MODEL_HORIZONS if m >= h]
    if not covering:
        raise ValueError(f"No model in MODEL_HORIZONS {MODEL_HORIZONS} covers +{h} minutes")
    horizon_to_model[h] = min(covering)
for m in MODEL_HORIZONS:
    print(f"  {m}-minute model -> columns {[f'+{h}' for h, mm in horizon_to_model.items() if mm == m]}")

# Models needed for the signal evaluation (section 8.4): shortest model covering each signal window
signal_window_to_model = {}
for w in SIGNAL_WINDOWS:
    covering = [m for m in MODEL_HORIZONS if m >= w]
    if covering:
        signal_window_to_model[w] = min(covering)
    else:
        print(f"⚠ No model covers the signal window of {w} minutes – skipped")

pred_frames, test_eval_rows, model_paths = [], [], {}
for m in MODEL_HORIZONS:
    horizons = [h for h, mm in horizon_to_model.items() if mm == m]
    if not horizons and m not in signal_window_to_model.values():
        continue
    path = model_latest_path(m)
    if not path.exists():
        raise FileNotFoundError(f"Model for {m} minutes not found: {path.resolve()} – train it first (section 6).")
    frame, rows, model_paths[m] = forecast_with_model(path, df_test_raw, horizons)
    pred_frames.append(frame)
    test_eval_rows += rows

pred_df = pred_frames[0]
for frame in pred_frames[1:]:
    pred_df = pred_df.merge(frame, on="_bar", how="outer", validate="one_to_one")
pred_cols = [
    f"Pred_Close_+{h}" if q == 0.5 else f"Pred_Close_+{h}_q{round(q * 100):02d}"
    for h in PREDICTION_HORIZONS for q in EXPORT_QUANTILES
]

test_output = test_csv_text.copy()
test_output["_bar"] = pd.to_numeric(test_output["Bar"]).astype("int64")
test_output = test_output.merge(pred_df[["_bar"] + pred_cols], on="_bar", how="left", validate="many_to_one")
test_output = test_output.drop(columns="_bar")
test_output.to_csv(TEST_OUTPUT_CSV_PATH, sep=";", index=False, encoding="utf-8-sig")

print(f"✓ Written: {TEST_OUTPUT_CSV_PATH}")
print(f"  {len(test_output):,} rows, {test_output[pred_cols[0]].notna().sum():,} with forecast, new columns: {pred_cols}")
display(test_output[["Bar", "Time UTC", "Close"] + pred_cols].tail(5))

### 8.3 Quick Check Against Actual Prices

Per horizon: model error vs. the naive forecast "price stays where it is" (**MAE/naive < 1 = model is better**) and the share
of correctly predicted directions (candles without price change are ignored).

In [ ]:
display(pd.DataFrame(test_eval_rows).sort_values("Horizon (min)").round(3))

### 8.4 Signal Evaluation in Ticks – Model vs. Random

Simulated trading on the test data, **one position at a time**, identical rules for the model and the comparisons:

* **Signal:** the median forecast path of the next `window` minutes moves at least `threshold` ticks away from the current
  close (largest predicted move up or down). Direction = direction of that move.
* **Entry:** close of the signal candle. **No entry** if the window contains a pause longer than `MAX_GAP_MINUTES`
  (session break, weekend).
* **Target:** the predicted price. Filled only if the price trades `TARGET_FILL_EXTRA_TICKS` beyond it within the window –
  e.g. reached in minute 13 of 15 counts.
* **Stop loss:** `STOP_LOSS_TICKS` against the entry. If stop and target are touched in the same candle, the stop counts.
* **Otherwise** closed at the close after `window` minutes. **Net** = gross − `COST_PER_TRADE_TICKS`.

**Comparisons** with exactly the same entry candles and target distances:
* *Random direction* – coin flip instead of the model direction (average of `BASELINE_RANDOM_RUNS` runs)
* *Always long* – shows how much comes from the gold uptrend alone

The model is only useful if it is **clearly better than both comparisons after costs**.

Key figures: profit factor = sum of winning ÷ sum of losing trades (net); t-value = Ø net ÷ std × √trades
(≥ 2 → unlikely to be chance); max drawdown in USD per contract.

In [ ]:
def predicted_moves(path: dict, window: int):
    """Largest predicted move (ticks) and its direction within the first `window` minutes of every forecast."""
    med = path["median_path"][:, :window]
    origin = path["origin_idx"]
    c0 = path["close"][origin][:, None]
    up = (med - c0).max(axis=1)
    down = (med - c0).min(axis=1)
    direction = np.where(up >= -down, 1, -1)
    move = np.where(direction == 1, up, -down)
    return direction, move / TICK_SIZE


def window_has_gap(path: dict, window: int) -> np.ndarray:
    """True for every forecast whose window contains a pause longer than MAX_GAP_MINUTES."""
    n = path["n_real_rows"]
    ts = pd.to_datetime(pd.Series(path["timestamp"][:n]))
    gap = (ts.diff().dt.total_seconds().div(60).fillna(0).to_numpy() > MAX_GAP_MINUTES).astype(int)
    cum = np.concatenate([[0], np.cumsum(gap)])            # cum[i] = number of gaps before candle i
    origin = path["origin_idx"]
    end = np.minimum(origin + window, n - 1)
    return (cum[end + 1] - cum[origin + 1]) > 0


def simulate_signals(path: dict, window: int, threshold_ticks: float, mode: str = "model", seed: int = 0) -> dict:
    """
    One position at a time. mode: "model" (model direction), "random" (coin flip), "long" (always long).
    Entries and target distances are always those of the model signals.
    """
    origin = path["origin_idx"]
    close, high, low = path["close"], path["high"], path["low"]
    direction, move_ticks = predicted_moves(path, window)
    if mode == "random":
        direction = np.random.default_rng(seed).choice([-1, 1], size=len(direction))
    elif mode == "long":
        direction = np.ones_like(direction)
    valid = (origin + window < path["n_real_rows"]) & ~window_has_gap(path, window)
    candidates = np.flatnonzero(valid & (move_ticks >= threshold_ticks))
    candidates = candidates[np.argsort(origin[candidates], kind="stable")]

    extra = TARGET_FILL_EXTRA_TICKS * TICK_SIZE
    trades, next_free = [], -1
    for r in candidates:
        t = int(origin[r])
        if t < next_free:                                   # still in a position
            continue
        d, target_ticks = int(direction[r]), float(move_ticks[r])
        entry = close[t]
        target = entry + d * target_ticks * TICK_SIZE
        hi, lo = high[t + 1:t + window + 1], low[t + 1:t + window + 1]
        if d == 1:
            tgt = hi >= target + extra
            stp = lo <= entry - STOP_LOSS_TICKS * TICK_SIZE if STOP_LOSS_TICKS else np.zeros(window, bool)
        else:
            tgt = lo <= target - extra
            stp = hi >= entry + STOP_LOSS_TICKS * TICK_SIZE if STOP_LOSS_TICKS else np.zeros(window, bool)
        k_tgt = int(np.argmax(tgt)) + 1 if tgt.any() else window + 1
        k_stp = int(np.argmax(stp)) + 1 if stp.any() else window + 1
        if k_stp <= k_tgt and k_stp <= window:              # stop first (or in the same candle -> conservative)
            k, exit_type, gross = k_stp, "stop", -float(STOP_LOSS_TICKS)
        elif k_tgt <= window:
            k, exit_type, gross = k_tgt, "target", target_ticks
        else:
            k, exit_type, gross = window, "time", d * (close[t + window] - entry) / TICK_SIZE
        adverse = (entry - lo[:k].min()) if d == 1 else (hi[:k].max() - entry)
        trades.append({"t": t, "direction": d, "target_ticks": target_ticks, "exit": exit_type,
                       "minutes": k, "gross_ticks": gross, "adverse_ticks": max(adverse, 0.0) / TICK_SIZE})
        next_free = t + k
    return {"signal_candles": len(candidates), "trades": pd.DataFrame(trades)}


def summarize_trades(trades: pd.DataFrame, timestamps, trading_days: int) -> dict:
    """Key figures of one simulation run."""
    if trades.empty:
        return {"Trades": 0}
    net = trades["gross_ticks"].to_numpy() - COST_PER_TRADE_TICKS
    equity = np.cumsum(net * TICK_VALUE_USD)
    drawdown = (np.maximum.accumulate(np.concatenate([[0.0], equity])) - np.concatenate([[0.0], equity])).max()
    wins, losses = net[net > 0].sum(), -net[net < 0].sum()
    months = pd.Series(net, index=pd.to_datetime(pd.Series(timestamps[trades["t"].to_numpy()])).dt.strftime("%Y-%m"))
    monthly = months.groupby(level=0).sum()
    is_long = trades["direction"].to_numpy() == 1
    return {
        "Trades": len(trades),
        "Trades/day": len(trades) / trading_days,
        "Long %": 100 * is_long.mean(),
        "Target %": 100 * (trades["exit"] == "target").mean(),
        "Stop %": 100 * (trades["exit"] == "stop").mean(),
        "Time exit %": 100 * (trades["exit"] == "time").mean(),
        "Ø net (ticks)": net.mean(),
        "Ø net long": net[is_long].mean() if is_long.any() else np.nan,
        "Ø net short": net[~is_long].mean() if (~is_long).any() else np.nan,
        "Profit factor": wins / losses if losses > 0 else np.inf,
        "t-value": net.mean() / net.std(ddof=1) * np.sqrt(len(net)) if len(net) > 1 and net.std(ddof=1) > 0 else np.nan,
        "Months in profit": f"{int((monthly > 0).sum())}/{len(monthly)}",
        "Net total USD/contract": net.sum() * TICK_VALUE_USD,
        "Max drawdown USD": drawdown,
        "Worst trade (ticks)": net.min(),
    }


signal_rows = []
for window, m in signal_window_to_model.items():
    path = model_paths[m]
    n_real = path["n_real_rows"]
    timestamps = path["timestamp"]
    trading_days = max(pd.to_datetime(pd.Series(timestamps[:n_real])).dt.date.nunique(), 1)

    _, move_ticks = predicted_moves(path, window)
    valid = path["origin_idx"] + window < n_real
    pct = np.percentile(move_ticks[valid], [50, 90, 99, 99.9])
    print(f"[{window}-minute window, {m} min model] predicted move in ticks: median {pct[0]:.1f}, "
          f"90% {pct[1]:.1f}, 99% {pct[2]:.1f}, 99.9% {pct[3]:.1f}, max {move_ticks[valid].max():.1f}")

    for threshold in SIGNAL_THRESHOLDS_TICKS:
        runs = {
            "Model": [simulate_signals(path, window, threshold, "model")],
            f"Random direction (Ø {BASELINE_RANDOM_RUNS})": [
                simulate_signals(path, window, threshold, "random", seed=s) for s in range(BASELINE_RANDOM_RUNS)
            ],
            "Always long": [simulate_signals(path, window, threshold, "long")],
        }
        for strategy, results in runs.items():
            summaries = [summarize_trades(res["trades"], timestamps, trading_days) for res in results]
            row = {"Window (min)": window, "Signal ≥ ticks": threshold, "Strategy": strategy,
                   "Signal candles": results[0]["signal_candles"]}
            if len(summaries) == 1:
                row.update(summaries[0])
            else:                                            # average of the random runs
                for key in summaries[0]:
                    values = [s.get(key) for s in summaries]
                    row[key] = (np.mean([v for v in values if isinstance(v, (int, float, np.floating))])
                                if not isinstance(values[0], str) else values[0])
            signal_rows.append(row)

signal_df = pd.DataFrame(signal_rows)
print(f"Rules: stop {STOP_LOSS_TICKS} ticks, target fill +{TARGET_FILL_EXTRA_TICKS} tick, costs {COST_PER_TRADE_TICKS} ticks "
      f"per round trip, tick value {TICK_VALUE_USD} USD, no trades across pauses > {MAX_GAP_MINUTES} min")
print(f"Test period: {pd.Timestamp(timestamps[0])} → {pd.Timestamp(timestamps[n_real - 1])}, {trading_days} trading days")
display(signal_df.round(2))